# Validación externa de la hipótesis — EMPS Fresnillo

Reproduce con scikit-learn el análisis que el sistema EMPS Fresnillo corre internamente. Sirve como **doble verificación**: si los números coinciden (±0.01), la implementación interna funciona como una librería estándar.

**Hipótesis**: "Una estimación temprana mejora cuando integra esfuerzo técnico, modo de desarrollo, cambios y viabilidad fiscal-laboral antes de comprometer precio, calendario y mantenimiento."

**Variable dependiente**: `mape_hours` (Mean Absolute Percentage Error).
**Umbral IFPUG**: ≤15% preciso · 15-30% aceptable · >30% impreciso.

## Pasos previos
1. Descarga el CSV desde `/investigacion/validacion-hipotesis` (botón **Descargar CSV**).
2. Guárdalo como `emps-dataset.csv` en la misma carpeta que este notebook.
3. `pip install pandas scikit-learn matplotlib seaborn`
4. Ejecuta todas las celdas (Cell → Run All).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy import stats

CSV_PATH = 'emps-dataset.csv'
df = pd.read_csv(CSV_PATH)
print(f'Filas cargadas: {len(df)}')
df.head()

## 1. Estadística descriptiva del MAPE

In [ ]:
print(df[['mape_hours','mape_cost']].describe())
accuracy_rate = (df['mape_hours'] <= 15).mean()
print(f'\nTasa de proyectos precisos (MAPE ≤ 15%): {accuracy_rate*100:.1f}%')

df['mape_hours'].hist(bins=20)
plt.axvline(15, color='r', linestyle='--', label='Umbral 15% (IFPUG)')
plt.xlabel('MAPE horas (%)'); plt.ylabel('Frecuencia'); plt.legend(); plt.title('Distribución del MAPE'); plt.show()

## 2. Construcción de features (igual que el sistema)

El sistema usa estas 10 features: clarity_avg, n_modules, n_integrations, criticality_avg, changes_anticipated_ratio, fiscal_detailed + 4 one-hot de dev_mode.

In [ ]:
features_num = ['clarity_avg', 'n_modules', 'n_integrations', 'criticality_avg', 'changes_anticipated_ratio', 'fiscal_detailed']
dev_modes = ['ai_assisted', 'hybrid', 'bytecoding_prompts', 'low_code']
for m in dev_modes:
    df[f'dev_mode_{m}'] = (df['dev_mode'] == m).astype(int)
features = features_num + [f'dev_mode_{m}' for m in dev_modes]
X = df[features].values
y = df['mape_hours'].values
y_binary = (df['mape_hours'] <= 15).astype(int)
print(f'Shape X: {X.shape}, Shape y: {y.shape}')

## 3. Correlación de Pearson de cada feature con MAPE

In [ ]:
corr_rows = []
for f in features:
    r, p = stats.pearsonr(df[f], df['mape_hours'])
    corr_rows.append({'feature': f, 'r': r, 'p_value': p, 'significativo': p < 0.05})
pd.DataFrame(corr_rows).sort_values('r')

## 4. Regresión multivariable lineal — EVIDENCIA PRINCIPAL

Compara R² con el valor reportado por el sistema interno. Debe coincidir (±0.01).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
reg = LinearRegression().fit(X, y)
y_pred = reg.predict(X)
r2 = r2_score(y, y_pred); rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f'R² = {r2:.4f}')
print(f'RMSE = {rmse:.2f}')
print(f'Intercept = {reg.intercept_:.2f}')
coefs = pd.DataFrame({'feature': features, 'coef': reg.coef_})
coefs.sort_values('coef')

## 5. Random Forest — clasificación 'precisa sí/no'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.3, random_state=42)
rf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=6).fit(X_train, y_train)
acc = rf.score(X_test, y_test)
print(f'Accuracy en test: {acc*100:.1f}%')
importance = pd.DataFrame({'feature': features, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
importance.plot.barh(x='feature', y='importance', figsize=(8,5), title='Feature importance — Random Forest')
plt.tight_layout(); plt.show()

## 6. Veredicto de la hipótesis (mismo umbral que el sistema)

- **Cumplida**: R² ≥ 0.35 Y ≥2 predictores con p < 0.05
- **Parcialmente cumplida**: R² entre 0.15 y 0.35, O 1 predictor significativo
- **No cumplida**: R² < 0.15 Y ningún predictor significativo
- **Datos insuficientes**: N < 15

In [ ]:
n = len(df)
sig_count = sum(1 for row in corr_rows if row['p_value'] < 0.05 and abs(row['r']) > 0.1)
if n < 15:
    veredicto = 'datos_insuficientes'
elif r2 >= 0.35 and sig_count >= 2:
    veredicto = 'CUMPLIDA'
elif r2 >= 0.15 or sig_count >= 1:
    veredicto = 'parcialmente cumplida'
else:
    veredicto = 'NO cumplida'
print(f'Veredicto: {veredicto}')
print(f'  N = {n}')
print(f'  R² = {r2:.4f}')
print(f'  Predictores significativos = {sig_count}')